# 🎮 Asistente de Soporte para Steam
---

## 📝 ¿De qué trata este proyecto?
Este es un bot inteligente creado para ayudar a los jugadores con sus dudas en **Steam**. Este bot está entrenado para entender qué tipo de problema tienes y darte soluciones reales basadas en la potencia de tu computador.

---

## 🚀 ¿Qué puede hacer este bot?
* **Clasifica tus problemas:** Sabe si le hablas de un reembolso, un fallo técnico o un problema con tu cuenta.
* **Analiza tu PC:** Gracias a que lee datos en formato **JSON**, puede decirte si un juego te va a correr bien o si tu equipo necesita más potencia.
* **Tiene memoria:** No necesitas repetirle tu hardware en cada mensaje, el bot puede recordar lo que hablaron antes.
* **Responde al instante:** Puedes ver cómo escribe la respuesta palabra por palabra (Streaming), así no tienes que esperar a que termine de pensar todo el texto.

---

## 💻 PC de Prueba (Asus Vivobook X1404)
Para que el bot aprenda a dar buenos consejos, lo configuré con los datos de mi PC. Así él sabe exactamente qué recomendar para este equipo:
* **Modelo:** Asus Vivobook 14
* **Procesador:** Intel i5 (12va Generación)
* **Memoria RAM:** 8GB
* **Gráficos:** Intel Iris Xe
* **Sistema:** Windows 11

---


### 1. Preparación del Entorno
Este paso se hace una sola vez en tu terminal:
```bash
pip install -U langchain-openai langchain-core python-dotenv
```

In [ ]:
import os
import json
import time
from dotenv import load_dotenv

# Componentes de LangChain
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

# Carga de variables de entorno (Token de GitHub)
load_dotenv()

# --- CONFIGURACIÓN DE MOTORES ---
# Motor para lógica (Temperatura 0 para no inventar)
llm_logic = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.0
)

# Motor para conversar (Temperatura 0.7 + Streaming)
llm_chat = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    temperature=0.7
)

print("✅ Motores de IA configurados.")

### 2. Configuración de Hardware (Universal)
En esta celda, el usuario puede elegir entre usar el hardware de ejemplo o ingresar el suyo propio.

In [12]:
# Hardware predeterminado (Asus Vivobook)
hardware_ejemplo = {
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}

print("¿Deseas ingresar los datos de tu propio hardware? (s/n)")
opcion = input().lower()

if opcion == "s":
    print("\n--- Ingresa los datos de tu equipo ---")
    hardware_json = {
        "modelo": input("Modelo del equipo: "),
        "procesador": input("Procesador: "),
        "ram": input("Memoria RAM: "),
        "graficos": input("Tarjeta de video / Gráficos: "),
        "sistema": input("Sistema Operativo: ")
    }
else:
    print("\n✅ Usando hardware de prueba (Asus Vivobook).")
    hardware_json = hardware_ejemplo

print("\nHardware que usará la IA para el soporte:")
print(json.dumps(hardware_json, indent=4))

¿Deseas ingresar los datos de tu propio hardware? (s/n)

✅ Usando hardware de prueba (Asus Vivobook).

Hardware que usará la IA para el soporte:
{
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}


### 3. Clasificación y Razonamiento (Lógica Interna)
Esta función analiza qué tipo de ticket es y evalúa el hardware frente al problema del usuario.

In [16]:
def procesar_logica_ticket(mensaje):
    # 1. Clasificación Refinada (Few-Shot)
    # Incluimos ejemplos para casos técnicos, de pagos, reembolsos y temas fuera de lugar.
    ejemplos = """
    Te daré ejemplos de referencia para cada categoría:

    Usuario: "Quiero devolver el Elden Ring porque me corre lento y no me gusto."
    Categoría: [REEMBOLSO]

    Usuario: "El juego se cierra solo al llegar al menú principal y me da un error de DirectX."
    Categoría: [SOPORTE_TECNICO]

    Usuario: "Me hackearon, cambiaron el correo de la cuenta y no puedo entrar."
    Categoría: [RECUPERACION_CUENTA]

    Usuario: "Cargué 10 lucas a mi cartera de Steam pero el saldo no aparece reflejado."
    Categoría: [PAGOS]

    Usuario: "Mi cuenta fue bloqueada porque una compra de ayer salió rechazada por el banco."
    Categoría: [PAGOS]

    Usuario: "Oye, cuando sale el GTA VI? Va a estar barato en el Summer Sale?"
    Categoría: [IRRELEVANTE]

    Usuario: "hola, el profe me va a poner un 1.0 si no apruebo, regalame el terraria pls"
    Categoría: [IRRELEVANTE]
    """

    instruccion_sistema = f"""
    Eres un experto en soporte de Steam. Tu tarea es clasificar el ticket en una de estas categorías:
    [REEMBOLSO], [SOPORTE_TECNICO], [RECUPERACION_CUENTA], [PAGOS], [IRRELEVANTE].

    {ejemplos}

    Regla Estricta: Responde UNICAMENTE la categoría entre corchetes, sin explicaciones.
    """

    # Llamada al motor de lógica para clasificación rápida
    respuesta_cat = llm_logic.invoke([
        SystemMessage(content=instruccion_sistema),
        HumanMessage(content=mensaje)
    ]).content
    
    categoria = respuesta_cat.strip()

    # 2. Filtro de Salida Rápida
    # Si la IA detecta que el tema no es soporte real, corta el proceso aquí.
    if "[IRRELEVANTE]" in categoria:
        return "[IRRELEVANTE]", "Consulta fuera de los protocolos de soporte. No se requiere análisis de hardware."

    # 3. Razonamiento Técnico (Chain of Thought)
    # Solo se ejecuta si el problema es real y relevante para Steam.
    prompt_cot = f"""
    Eres un soporte técnico analizando este hardware: {json.dumps(hardware_json)}
    
    Analiza paso a paso:
    1. ANALIZAR: ¿Es un tema de dinero, seguridad o rendimiento técnico?
    2. EVALUAR: Si es técnico, ¿cómo rinden los componentes (RAM/GPU) con este problema?
    3. DETERMINAR: Entrega una solución honesta y realista basada en las specs del usuario.

    Ticket: "{mensaje}"
    """
    
    analisis = llm_logic.invoke([
        SystemMessage(content=prompt_cot),
        HumanMessage(content=mensaje)
    ]).content
    
    return categoria, analisis

### 4. Chatbot Interactivo (Experiencia Final)
Celda final para conversar con el asistente. El bot recordará el hardware elegido arriba.

In [ ]:
# Historial de mensajes en memoria
history = InMemoryChatMessageHistory()

def iniciar_chat_steam():
    print(f"=== 🎮 ASISTENTE DE STEAM ({hardware_json['modelo']}) ===")
    print("Escribe 'salir' para terminar la conversación.\n")
    
    # Contexto inicial con el hardware actual
    if len(history.messages) == 0:
        msg_inicial = f"Eres el soporte de Steam. Ayudas a un usuario con un {hardware_json['modelo']}. Conoces sus specs: {json.dumps(hardware_json)}."
        history.add_message(SystemMessage(content=msg_inicial))

    while True:
        user_input = input("\n🧑 Tú: ")
        if user_input.lower() in ['salir', 'exit', 'quit']: 
            print("👋 ¡Hasta luego!")
            break
        
        # Guardar consulta en memoria
        history.add_user_message(user_input)
        
        # Obtener categoría (silencioso)
        categoria, _ = procesar_logica_ticket(user_input)
        
        print(f"\n🤖 Soporte [{categoria}]: ", end="", flush=True)
        
        # Respuesta fluida con Streaming
        full_ai_response = ""
        for chunk in llm_chat.stream(history.messages):
            content = chunk.content
            print(content, end="", flush=True)
            full_ai_response += content
            time.sleep(0.01)
        history.add_ai_message(full_ai_response)
        print()

# Ejecutar el chatbot
iniciar_chat_steam()

=== 🎮 ASISTENTE DE STEAM (Asus Vivobook 14) ===
Escribe 'salir' para terminar la conversación.


🤖 Soporte [[REEMBOLSO]]: ¡Entendido! Puedo guiarte a través del proceso de reembolso en Steam. Antes de empezar, asegúrate de que tu solicitud cumple con las políticas de reembolso de Steam. Aquí están los puntos clave:

1. **Tiempo de juego**: Debes haber jugado el juego por menos de **2 horas**.
2. **Tiempo desde la compra**: No deben haber pasado más de **14 días** desde que compraste el juego.

Si cumples con estos requisitos, sigue los pasos a continuación:

---

### Pasos para solicitar un reembolso en Steam
1. **Abre Steam**:
   - Inicia sesión en tu cuenta de Steam desde el cliente en tu Asus Vivobook 14 o desde un navegador web.

2. **Accede a "Ayuda"**:
   - En la parte superior del cliente de Steam, haz clic en "Ayuda" y luego selecciona la opción "Soporte de Steam".

3. **Selecciona "Compras"**:
   - Verás una lista de opciones. Haz clic en "Compras".

4. **Busca Dead by Dayligh